# Mid Progressive Stream 1
SDS(RGB) + LLVIP(Thermal)

In [ ]:
import subprocess, sys
for pkg in ['ultralytics', 'pandas', 'matplotlib', 'ensemble_boxes', 'optuna']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Packages OK')

In [ ]:
import os, sys, gc, math, time, random, json
sys.path.insert(0, '/root/AIP491')
from train_common import *

print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# === Kiem tra cau truc thu muc data ===
import os
def show_tree(path, depth=3, _d=0):
    if _d > depth or not os.path.exists(path):
        return
    entries = sorted(os.listdir(path))
    files = [e for e in entries if os.path.isfile(os.path.join(path, e))]
    dirs  = [e for e in entries if os.path.isdir(os.path.join(path, e))]
    pad = '  ' * _d
    for d in dirs[:8]:
        print(f'{pad}[{d}/]')
        show_tree(os.path.join(path, d), depth, _d+1)
    if files:
        print(f'{pad}  ({len(files)} files, e.g. {files[0]})')

print(f'RGBT_DATA_DIR = {RGBT_DATA_DIR}')
show_tree(RGBT_DATA_DIR, depth=3)

In [ ]:
STREAM = 1
RGB_BACKBONE_PATH = STREAM_CONFIGS[STREAM]['rgb']
THERMAL_BACKBONE_PATH = STREAM_CONFIGS[STREAM]['thr']
OUTPUT_DIR = os.path.join(BASE_DIR, 'Mid-stage', 'outputs', 'progressive_stream1')
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEEDS = [42, 777, 123]
BATCH_SIZE = 32
STAGE1_EPOCHS = 5;  STAGE1_LR = 1e-5
STAGE2_EPOCHS = 10; STAGE2_LR = 1e-4; STAGE2_UNFREEZE = 3
STAGE3_EPOCHS = 20; STAGE3_LR = 5e-4; STAGE3_PATIENCE = 2

print(f'Stream {STREAM}: {STREAM_CONFIGS[STREAM]["desc"]}')

In [ ]:
class RGBTFusionDetector(nn.Module):
    EXTRACT_LAYERS = {4: 64, 6: 128, 9: 256}

    def __init__(self, rgb_backbone, thermal_backbone, nc=1, freeze_backbones=True):
        super().__init__()
        self.rgb_stream = rgb_backbone
        self.thermal_stream = thermal_backbone

        if freeze_backbones:
            for p in self.rgb_stream.parameters():
                p.requires_grad = False
            for p in self.thermal_stream.parameters():
                p.requires_grad = False

        self.fuse_p3 = nn.Sequential(nn.Conv2d(128, 128, 1, bias=False), nn.BatchNorm2d(128), nn.SiLU())
        self.fuse_p4 = nn.Sequential(nn.Conv2d(256, 256, 1, bias=False), nn.BatchNorm2d(256), nn.SiLU())
        self.fuse_p5 = nn.Sequential(nn.Conv2d(512, 512, 1, bias=False), nn.BatchNorm2d(512), nn.SiLU())

        self.upsample   = nn.Upsample(scale_factor=2, mode='nearest')
        self.td_c2f_p4  = C2f(512 + 256, 256, n=1, shortcut=False)
        self.td_c2f_p3  = C2f(256 + 128, 128, n=1, shortcut=False)
        self.bu_conv_p4 = Conv(128, 128, 3, 2)
        self.bu_c2f_p4  = C2f(128 + 256, 256, n=1, shortcut=False)
        self.bu_conv_p5 = Conv(256, 256, 3, 2)
        self.bu_c2f_p5  = C2f(256 + 512, 512, n=1, shortcut=False)

        self.detect = Detect(nc=nc, ch=(128, 256, 512))
        self.detect.stride = torch.tensor([8., 16., 32.])

    def unfreeze_backbone_layers(self, n_layers):
        total_layers = 10
        start_idx = max(0, total_layers - n_layers)
        unfrozen = 0
        for stream_name, stream in [('RGB', self.rgb_stream), ('Thermal', self.thermal_stream)]:
            for i, layer in enumerate(stream):
                if i >= start_idx:
                    for p in layer.parameters():
                        p.requires_grad = True
                    unfrozen += sum(p.numel() for p in layer.parameters())
        print(f'  Unfreeze {{n_layers}} layers (idx >= {{start_idx}}): {{unfrozen:,}} params')

    def _extract_features(self, stream, x):
        feats = {}
        for i, layer in enumerate(stream):
            x = layer(x)
            if i in self.EXTRACT_LAYERS:
                feats[i] = x
        return feats

    def forward(self, rgb, thermal):
        rgb_f = self._extract_features(self.rgb_stream, rgb)
        thr_f = self._extract_features(self.thermal_stream, thermal)

        p3 = self.fuse_p3(torch.cat([rgb_f[4], thr_f[4]], dim=1))
        p4 = self.fuse_p4(torch.cat([rgb_f[6], thr_f[6]], dim=1))
        p5 = self.fuse_p5(torch.cat([rgb_f[9], thr_f[9]], dim=1))

        p4_td  = self.td_c2f_p4(torch.cat([self.upsample(p5), p4], dim=1))
        p3_out = self.td_c2f_p3(torch.cat([self.upsample(p4_td), p3], dim=1))

        p4_out = self.bu_c2f_p4(torch.cat([self.bu_conv_p4(p3_out), p4_td], dim=1))
        p5_out = self.bu_c2f_p5(torch.cat([self.bu_conv_p5(p4_out), p5], dim=1))

        return self.detect([p3_out, p4_out, p5_out])

print('RGBTFusionDetector (Progressive) OK')

In [ ]:
# === Optuna Bayesian Tuning (15 trials) ===
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

_tune_out = os.path.join(OUTPUT_DIR, 'tune_best_params.json')

if os.path.exists(_tune_out):
    print(f'Tune results found: {_tune_out}')
    with open(_tune_out) as _f:
        _best = json.load(_f)
    STAGE1_LR = _best.get('stage1_lr', STAGE1_LR)
    STAGE2_LR = _best.get('stage2_lr', STAGE2_LR)
    STAGE3_LR = _best.get('stage3_lr', STAGE3_LR)
    BATCH_SIZE = _best.get('batch_size', BATCH_SIZE)
    print('Loaded params:', _best)
else:
    FUSION_DATA_DIR = os.path.join(BASE_DIR, 'data', 'mid_yolo')
    _fyd = setup_mid_dataset(FUSION_DATA_DIR)

    def _tune_prog(trial):
        _s1lr = trial.suggest_float('stage1_lr', 1e-6, 1e-4, log=True)
        _s2lr = trial.suggest_float('stage2_lr', 5e-5, 5e-4, log=True)
        _s3lr = trial.suggest_float('stage3_lr', 1e-4, 2e-3, log=True)
        _bs   = trial.suggest_categorical('batch_size', [16, 32])

        try:
            set_seed(42)
            _train_ds = RGBTDataset(_fyd, 'train', IMG_SIZE, blur_prob=0.0, flip_prob=0.5)
            _val_ds   = RGBTDataset(_fyd, 'val', IMG_SIZE)
            _train_ld = DataLoader(_train_ds, batch_size=_bs, shuffle=True,
                                   num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)
            _val_ld   = DataLoader(_val_ds, batch_size=_bs, shuffle=False,
                                   num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)

            _rgb_bb = load_yolov8n_backbone(RGB_BACKBONE_PATH)
            _thr_bb = load_yolov8n_backbone(THERMAL_BACKBONE_PATH)
            _model  = RGBTFusionDetector(_rgb_bb, _thr_bb, nc=1, freeze_backbones=True).to(device)

            # Stage 1: 3 epochs
            _opt = AdamW([p for p in _model.parameters() if p.requires_grad], lr=_s1lr, weight_decay=1e-3)
            _crit = v8DetectionLoss(LossModel(_model))
            for _ in range(3):
                run_epoch(_model, _train_ld, _opt, _crit, train=True)

            # Stage 2: unfreeze 3 layers, 3 epochs
            _model.unfreeze_backbone_layers(3)
            _opt = AdamW([p for p in _model.parameters() if p.requires_grad], lr=_s2lr, weight_decay=1e-3)
            _crit = v8DetectionLoss(LossModel(_model))
            for _ in range(3):
                run_epoch(_model, _train_ld, _opt, _crit, train=True)

            # Stage 3: full unfreeze, 5 epochs
            _model.unfreeze_backbone_layers(10)
            _opt = AdamW([p for p in _model.parameters() if p.requires_grad], lr=_s3lr, weight_decay=1e-3)
            _crit = v8DetectionLoss(LossModel(_model))
            _best_val = float('inf')
            for _ep in range(5):
                run_epoch(_model, _train_ld, _opt, _crit, train=True)
                _vl = run_epoch(_model, _val_ld, _opt, _crit, train=False)
                if _vl < _best_val:
                    _best_val = _vl

            del _model, _rgb_bb, _thr_bb, _opt, _crit
            torch.cuda.empty_cache(); gc.collect()
            return _best_val

        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                print(f'  OOM at bs={_bs}, skip trial')
                torch.cuda.empty_cache(); gc.collect()
                return float('inf')
            raise

    _study = optuna.create_study(
        direction='minimize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3),
        study_name='progressive_tune'
    )
    _study.optimize(_tune_prog, n_trials=15, show_progress_bar=True)

    _best = dict(_study.best_params)
    _best['best_val_loss'] = _study.best_value
    with open(_tune_out, 'w') as _f:
        json.dump(_best, _f, indent=2)

    STAGE1_LR = _best.get('stage1_lr', STAGE1_LR)
    STAGE2_LR = _best.get('stage2_lr', STAGE2_LR)
    STAGE3_LR = _best.get('stage3_lr', STAGE3_LR)
    BATCH_SIZE = _best.get('batch_size', BATCH_SIZE)
    print('BEST PARAMS:')
    for _k, _v in _best.items():
        print(f'  {_k}: {_v}')

In [ ]:
# === Training (3-stage progressive unfreezing) ===
# Augmentation: Random Gaussian Blur (chi RGB, p=0.5) + Random HFlip (ca 2, p=0.5)
FUSION_DATA_DIR = os.path.join(BASE_DIR, 'data', 'mid_yolo')
fusion_yolo_dir = setup_mid_dataset(FUSION_DATA_DIR)

train_ds = RGBTDataset(fusion_yolo_dir, 'train', IMG_SIZE, blur_prob=0.0, flip_prob=0.5)
val_ds   = RGBTDataset(fusion_yolo_dir, 'val', IMG_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)


def run_stage(model, train_loader, val_loader, lr, num_epochs,
              patience=None, use_cosine=False, stage_name=''):
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-3)
    if use_cosine:
        scheduler = LambdaLR(optimizer, lambda ep: 0.01 + 0.5 * (1 - 0.01) * (1 + math.cos(math.pi * ep / max(num_epochs, 1))))
    else:
        scheduler = LambdaLR(optimizer, lambda ep: 1.0)

    criterion = v8DetectionLoss(LossModel(model))
    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    best_val = float('inf')
    no_improve = 0
    best_state = None

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'\n  [{stage_name}] {num_epochs} epochs, lr={lr}, trainable={trainable:,}/{total:,} ({100*trainable/total:.1f}%)')

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        tr_loss = run_epoch(model, train_loader, optimizer, criterion, train=True)
        va_loss = run_epoch(model, val_loader, optimizer, criterion, train=False)
        scheduler.step()
        cur_lr = scheduler.get_last_lr()[0]

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['lr'].append(cur_lr)

        flag = ''
        if va_loss < best_val:
            best_val = va_loss
            no_improve = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            flag = '  <- best'
        else:
            no_improve += 1
            if patience:
                flag = f'  (no improve {no_improve}/{patience})'

        elapsed = time.time() - t0
        print(f'  [{stage_name}] Epoch {epoch:3d}/{num_epochs}  train={tr_loss:.4f}  val={va_loss:.4f}  lr={cur_lr:.2e}  {elapsed:.0f}s{flag}')

        if patience and no_improve >= patience:
            print(f'  [{stage_name}] Early stopping at epoch {epoch}')
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return history, best_val


all_histories = {}
all_results = {}

for seed_idx, SEED in enumerate(SEEDS):
    seed_dir = os.path.join(OUTPUT_DIR, f'seed_{SEED}')
    os.makedirs(seed_dir, exist_ok=True)
    best_path = os.path.join(seed_dir, 'fusion_best.pt')

    if os.path.exists(best_path):
        ckpt = torch.load(best_path, map_location=device, weights_only=False)
        if 'metrics' in ckpt:
            all_results[SEED] = ckpt['metrics']
        all_histories[SEED] = ckpt.get('history', {})
        print(f'Seed {SEED}: checkpoint exists, skip.')
        continue

    print(f'\n{"="*60}')
    print(f'  SEED {SEED} ({seed_idx+1}/{len(SEEDS)})')
    print(f'{"="*60}')

    set_seed(SEED)
    rgb_bb = load_yolov8n_backbone(RGB_BACKBONE_PATH)
    thr_bb = load_yolov8n_backbone(THERMAL_BACKBONE_PATH)
    fusion_model = RGBTFusionDetector(rgb_bb, thr_bb, nc=1, freeze_backbones=True).to(device)

    combined_history = {'train_loss': [], 'val_loss': [], 'lr': []}

    # Stage 1: backbone dong hoan toan
    h1, _ = run_stage(fusion_model, train_loader, val_loader,
                      lr=STAGE1_LR, num_epochs=STAGE1_EPOCHS,
                      patience=None, use_cosine=False, stage_name='Stage 1 - Frozen')
    for k in combined_history:
        combined_history[k].extend(h1[k])

    # Stage 2: mo 3 layer cuoi backbone
    fusion_model.unfreeze_backbone_layers(STAGE2_UNFREEZE)
    h2, _ = run_stage(fusion_model, train_loader, val_loader,
                      lr=STAGE2_LR, num_epochs=STAGE2_EPOCHS,
                      patience=None, use_cosine=False, stage_name='Stage 2 - Unfreeze 3')
    for k in combined_history:
        combined_history[k].extend(h2[k])

    # Stage 3: mo toan bo + cosine + early stopping
    fusion_model.unfreeze_backbone_layers(10)
    h3, best_val = run_stage(fusion_model, train_loader, val_loader,
                             lr=STAGE3_LR, num_epochs=STAGE3_EPOCHS,
                             patience=STAGE3_PATIENCE, use_cosine=True,
                             stage_name='Stage 3 - Full')
    for k in combined_history:
        combined_history[k].extend(h3[k])

    all_histories[SEED] = combined_history
    torch.save({
        'seed': SEED,
        'model_state_dict': fusion_model.state_dict(),
        'val_loss': best_val,
        'history': combined_history
    }, best_path)
    print(f'Seed {SEED}: done. Best val_loss = {best_val:.4f}')

    del fusion_model, rgb_bb, thr_bb
    torch.cuda.empty_cache(); gc.collect()

print(f'Training done for {len(SEEDS)} seeds.')

In [ ]:
# === Evaluation ===
for SEED in SEEDS:
    seed_dir = os.path.join(OUTPUT_DIR, f'seed_{SEED}')
    best_path = os.path.join(seed_dir, 'fusion_best.pt')
    if SEED in all_results:
        continue
    if not os.path.exists(best_path):
        continue

    print(f'\nEvaluating seed {SEED}...')
    set_seed(SEED)
    rgb_bb = load_yolov8n_backbone(RGB_BACKBONE_PATH)
    thr_bb = load_yolov8n_backbone(THERMAL_BACKBONE_PATH)
    fusion_model = RGBTFusionDetector(rgb_bb, thr_bb, nc=1, freeze_backbones=False).to(device)

    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    fusion_model.load_state_dict(ckpt['model_state_dict'])

    metrics = evaluate_model(fusion_model, val_loader)
    all_results[SEED] = metrics
    ckpt['metrics'] = metrics
    torch.save(ckpt, best_path)

    print(f'  mAP@0.5: {metrics["map50"]:.4f} | mAP@.5:.95: {metrics["map50_95"]:.4f}')
    del fusion_model, rgb_bb, thr_bb
    torch.cuda.empty_cache(); gc.collect()

print_summary(all_results, SEEDS, title=f'Mid Progressive Stream {STREAM} -- {STREAM_CONFIGS[STREAM]["desc"]}')
plot_loss_curves(all_histories, SEEDS,
                 os.path.join(OUTPUT_DIR, 'loss_curves.png'),
                 title=f'Mid Progressive Stream {STREAM}')